In [3]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
from functions.nba_clean_data import archive_files
from functions.nba_proj_scape import nba_proj_scape

In [4]:
# %load functions/get_nba_spread

In [5]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nba?site=fanduel"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nba/odds"})
odds = [i.text for i in odds_list]
teams.sort()
print(teams)

['ATL', 'BOS', 'CHA', 'CHI', 'DAL', 'DET', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'NYK', 'OKC', 'PHO', 'POR', 'SAC', 'SAS', 'UTA']


In [6]:
df_fd = pd.read_csv('data/nba/FanDuel-NBA-2021 ET-12 ET-29 ET-69397-players-list.csv')
teams2 = list(df_fd.Team.unique())
teams2.sort()
print(teams2)

['ATL', 'BOS', 'CHA', 'CHI', 'DAL', 'DET', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'NY', 'OKC', 'PHO', 'POR', 'SA', 'SAC', 'UTA']


In [7]:
replace_dict = {"GSW":"GS", "NYK":"NY", "NOP":"NO", "SAS":"SA"}
clean_teams = [replace_dict.get(item,item) for item in teams]
print(clean_teams)

['ATL', 'BOS', 'CHA', 'CHI', 'DAL', 'DET', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'NY', 'OKC', 'PHO', 'POR', 'SAC', 'SA', 'UTA']


In [8]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'DAL': '102.25', 'OKC': '102.50', 'CHI': '103.00', 'MIA': '105.25', 'DET': '108.75', 'LAL': '109.50', 'UTA': '109.50', 'IND': '109.75', 'SAC': '110.50', 'CHA': '111.00', 'SA': '111.00', 'ATL': '111.25', 'NY': '111.75', 'BOS': '115.25', 'LAC': '115.75', 'PHO': '117.00', 'MEM': '117.50', 'POR': '117.50'}


In [9]:
stats_df = nba_proj_scape()
stats_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Nickname  230 non-null    object
 1   FP        230 non-null    object
 2   Value     230 non-null    object
 3   Min       230 non-null    object
dtypes: object(4)
memory usage: 7.3+ KB


In [10]:
try:
    stats_df.replace("Mohamed Bamba", "Mo Bamba", regex=True, inplace=True)
    stats_df.query("Nickname == 'Mo Bamba'")
except:
    print('no mo bamba')

In [11]:
df_fd = pd.merge(df_fd, stats_df, how='left', on='Nickname')
df_fd

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,FP,Value,Min
0,69397-9488,SF/PF,LeBron,LeBron James,James,50.717391,23.0,11200,LAL@MEM,LAL,MEM,GTD,Abdomen,NaN,NaN,NaN,SF/PF,52.8,4.71,38.29
1,69397-67708,PG/SG,Dejounte,Dejounte Murray,Murray,45.961292,31.0,10200,MIA@SA,SA,MIA,O,Postponed,NaN,NaN,NaN,PG/SG,NaN,NaN,NaN
2,69397-84669,PG/SG,Luka,Luka Doncic,Doncic,48.076189,21.0,10000,DAL@SAC,DAL,SAC,O,Covid-19,NaN,NaN,NaN,PG/SG,NaN,NaN,NaN
3,69397-80808,SF/PF,Jayson,Jayson Tatum,Tatum,44.212121,33.0,9900,LAC@BOS,BOS,LAC,O,Covid-19,NaN,NaN,NaN,SF/PF,NaN,NaN,NaN
4,69397-9642,PG,Russell,Russell Westbrook,Westbrook,41.462856,35.0,9800,LAL@MEM,LAL,MEM,NaN,NaN,NaN,NaN,NaN,PG,44.3,4.52,36.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,69397-145354,SG,Devon,Devon Dotson,Dotson,6.866667,6.0,3500,ATL@CHI,CHI,ATL,NaN,NaN,NaN,NaN,NaN,SG,0.0,0,0.05
345,69397-101330,SF,Emanuel,Emanuel Terry,Terry,NaN,NaN,3500,OKC@PHO,PHO,OKC,NaN,NaN,NaN,NaN,NaN,SF,0.0,0,0.02
346,69397-49115,SG,Wayne,Wayne Selden,Selden,2.566667,3.0,3500,NY@DET,NY,DET,O,Covid-19,NaN,NaN,NaN,SG,NaN,NaN,NaN
347,69397-97263,C/PF,Drew,Drew Eubanks,Eubanks,13.843750,32.0,3500,MIA@SA,SA,MIA,O,Postponed,NaN,NaN,NaN,PF/C,NaN,NaN,NaN


In [12]:
pd.set_option('display.max_rows', df_fd.shape[0]+1)
cols = ['FP', 'Value', 'Min']
df_fd[cols] = df_fd[cols].astype(float).fillna(0.0)
df_fd = df_fd[df_fd.FP > 15.0]
df_fd

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,FP,Value,Min
0,69397-9488,SF/PF,LeBron,LeBron James,James,50.717391,23.0,11200,LAL@MEM,LAL,MEM,GTD,Abdomen,NaN,NaN,NaN,SF/PF,52.8,4.71,38.29
4,69397-9642,PG,Russell,Russell Westbrook,Westbrook,41.462856,35.0,9800,LAL@MEM,LAL,MEM,NaN,NaN,NaN,NaN,NaN,PG,44.3,4.52,36.16
5,69397-84671,PG,Trae,Trae Young,Young,45.253333,30.0,9700,ATL@CHI,ATL,CHI,NaN,NaN,NaN,NaN,NaN,PG,50.0,5.15,37.29
7,69397-145485,PG,LaMelo,LaMelo Ball,Ball,44.482759,29.0,9400,CHA@IND,CHA,IND,NaN,NaN,NaN,NaN,NaN,PG,44.9,4.78,33.65
9,69397-84680,SG/PG,Shai,Shai Gilgeous-Alexander,Gilgeous-Alexander,39.026668,30.0,9300,OKC@PHO,OKC,PHO,NaN,NaN,NaN,NaN,NaN,PG/SG,42.1,4.53,34.31
10,69397-9714,SG/SF,DeMar,DeMar DeRozan,DeRozan,41.282144,28.0,9200,ATL@CHI,CHI,ATL,NaN,NaN,NaN,NaN,NaN,SG/SF,42.6,4.63,34.06
11,69397-40203,C,Rudy,Rudy Gobert,Gobert,42.445453,33.0,9100,UTA@POR,UTA,POR,NaN,NaN,NaN,NaN,NaN,C,42.9,4.71,33.09
12,69397-20848,PG,Damian,Damian Lillard,Lillard,40.055556,27.0,9000,UTA@POR,POR,UTA,NaN,NaN,NaN,NaN,NaN,PG,47.7,5.30,38.08
13,69397-63122,PF/C,Kristaps,Kristaps Porzingis,Porzingis,38.966667,24.0,8900,DAL@SAC,DAL,SAC,NaN,NaN,NaN,NaN,NaN,PF/C,47.8,5.37,34.66
14,69397-67026,SF/SG,Jaylen,Jaylen Brown,Brown,35.675000,20.0,8800,LAC@BOS,BOS,LAC,NaN,NaN,NaN,NaN,NaN,SG/SF,42.4,4.82,36.21


In [13]:
df_fd['FPPG'] = df_fd.FP
dropped_cols = ['FP', 'Value', 'Min']#, 'expts', 'rkpts', 'vlpts', 'new_expts']
df_fd.drop(dropped_cols, axis=1, inplace=True)
df_fd

/tmp/ipykernel_5869/3747236435.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fd['FPPG'] = df_fd.FP
/home/michael/.local/share/virtualenvs/fan_fact-icAuygf9/lib/python3.8/site-packages/pandas/core/frame.py:4906: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().drop(


,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,69397-9488,SF/PF,LeBron,LeBron James,James,52.8,23.0,11200,LAL@MEM,LAL,MEM,GTD,Abdomen,NaN,NaN,NaN,SF/PF
4,69397-9642,PG,Russell,Russell Westbrook,Westbrook,44.3,35.0,9800,LAL@MEM,LAL,MEM,NaN,NaN,NaN,NaN,NaN,PG
5,69397-84671,PG,Trae,Trae Young,Young,50.0,30.0,9700,ATL@CHI,ATL,CHI,NaN,NaN,NaN,NaN,NaN,PG
7,69397-145485,PG,LaMelo,LaMelo Ball,Ball,44.9,29.0,9400,CHA@IND,CHA,IND,NaN,NaN,NaN,NaN,NaN,PG
9,69397-84680,SG/PG,Shai,Shai Gilgeous-Alexander,Gilgeous-Alexander,42.1,30.0,9300,OKC@PHO,OKC,PHO,NaN,NaN,NaN,NaN,NaN,PG/SG
10,69397-9714,SG/SF,DeMar,DeMar DeRozan,DeRozan,42.6,28.0,9200,ATL@CHI,CHI,ATL,NaN,NaN,NaN,NaN,NaN,SG/SF
11,69397-40203,C,Rudy,Rudy Gobert,Gobert,42.9,33.0,9100,UTA@POR,UTA,POR,NaN,NaN,NaN,NaN,NaN,C
12,69397-20848,PG,Damian,Damian Lillard,Lillard,47.7,27.0,9000,UTA@POR,POR,UTA,NaN,NaN,NaN,NaN,NaN,PG
13,69397-63122,PF/C,Kristaps,Kristaps Porzingis,Porzingis,47.8,24.0,8900,DAL@SAC,DAL,SAC,NaN,NaN,NaN,NaN,NaN,PF/C
14,69397-67026,SF/SG,Jaylen,Jaylen Brown,Brown,42.4,20.0,8800,LAC@BOS,BOS,LAC,NaN,NaN,NaN,NaN,NaN,SG/SF


In [14]:
df_fd.to_csv('data/nba/FD_proj.csv')

In [15]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [17]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack, RandomFantasyPointsStrategy

optimizer = get_optimizer(Site.FANDUEL, Sport.BASKETBALL)
optimizer.load_players_from_csv("data/nba/FD_proj.csv")

# p1 = optimizer.player_pool.get_player_by_name("Kevin Porter")
# p2 = optimizer.player_pool.get_player_by_name("Jarret Allen")
# p3 = optimizer.player_pool.get_player_by_name("Lonzo Ball")
# p4 = optimizer.player_pool.get_player_by_name("Evan Mobley")
# p5 = optimizer.player_pool.get_player_by_name("Isaiah Stewart")

# p1.max_exposure = 0.7
# p1.min_exposure = 0.3
# p2.max_exposure = 0.7 
# p2.min_exposure = 0.3
# p3.max_exposure = 0.65
# p3.min_exposure = 0.3
# p4.max_exposure = 0.65
# p4.min_exposure = 0.3
# p5.max_exposure = 0.6
# p5.min_exposure = 0.3     

# tm1 = 'ATL'
# tm2 = 'OKC'

# optimizer.add_stack(TeamStack(3, for_teams=[tm1,tm2], 
#                               max_exposure_per_team={tm1:0.6, tm2:0.55}))                                                                                                                                                                                                           

# optimizer.player_pool.remove_player('Zach LaVine')
# optimizer.set_max_repeating_players(5)
optimizer.set_fantasy_points_strategy(RandomFantasyPointsStrategy(max_deviation=0.1))
for lineup in optimizer.optimize(1, max_exposure=.5, exposure_strategy=AfterEachExposureStrategy):
    print(lineup)
optimizer.export('data/nba/fd_result.csv')

 1. PG      Damian Lillard                PG    POR            UTA@POR  47.7(46.467)   9000.0$   
 2. PG      De'Aaron Fox                  PG    SAC            DAL@SAC  37.1(40.169)   7400.0$   
 3. SG      Caris LeVert                  SF/SG IND            CHA@IND  37.0(40.225)   6600.0$   
 4. SG      Eric Bledsoe                  PG/SG LAC            LAC@BOS  30.6(33.135)   5500.0$   
 5. SF      Hamidou Diallo                SF/SG DET            NY@DET   33.4(35.153)   6000.0$   
 6. SF      Kyle Anderson                 PF/SF MEM            LAL@MEM  28.9(30.88)    5100.0$   
 7. PF      Domantas Sabonis              C/PF  IND            CHA@IND  44.4(46.227)   8400.0$   
 8. PF      Isaiah Roby                   C/PF  OKC            OKC@PHO  24.2(26.519)   3500.0$   
 9. C       Nikola Vucevic                C     CHI            ATL@CHI  42.1(44.821)   8300.0$   

Fantasy Points 325.40(343.60)
Salary 59800.00

